# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 MS dataset (Croissant package) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset schema
dataset = mlc.Dataset(croissant_url)

# Examine high-level metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"Version: {md.version} (Published: {getattr(md, 'datePublished', None)})")
print()
print("Fields included:")
for k in ['keywords', 'license', 'author', 'citation', 'conformsTo']:
    val = getattr(md, k, None)
    if val:
        print(f"  - {k}: {val}")

## 2. Data Overview
Review available record sets, data fields, and their `@id` values (unique Croissant IDs).

In [ ]:
# List all available record sets and fields with their @ids

all_record_sets = list(dataset.iter_record_sets())
print("Record Sets and their @id(s):\n")
for i, rs in enumerate(all_record_sets, 1):
    print(f"{i}. Name: {rs.name}")
    print(f"   @id: {rs.id}")
    # Show field info
    field_strs = []
    for field in rs.fields:
        dtype = getattr(field, 'dataType', None) or getattr(field, 'column', None)
        field_strs.append(f"      - {field.name} (@id={field.id}, type={dtype})")
    print("   Fields:\n" + ("\n".join(field_strs) or "      (none found)"))
    print()
if not all_record_sets:
    print("No record sets found! Check that your Croissant schema includes record sets.")

## 3. Data Extraction
Load data from available record sets into pandas DataFrames for inspection and analysis.

**Note:** All references to dataset entities (record sets, fields, columns) are made using their `@id` for full reproducibility and provenance.

In [ ]:
# Extract data for each record set, referenced by @id

record_sets_ids = [rs.id for rs in all_record_sets]
dataframes = {}

for recset_id in record_sets_ids:
    print(f"Loading records from record set: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"  - Loaded {len(df)} records; columns: {list(df.columns)[:8]}{' ...' if len(df.columns) > 8 else ''}")
    else:
        print("  - No records found!")

# Pick the first record set for display (customize this to target a specific record set by @id)
if record_sets_ids:
    example_rs_id = record_sets_ids[0]
    print(f"\nExample DataFrame for record set {example_rs_id}:")
    print(dataframes[example_rs_id].head() if example_rs_id in dataframes else "  (no data loaded)")

## 4. Exploratory Data Analysis (EDA)
Let's apply some data processing steps—filtering, normalization, and grouping—on a sample numeric field. All field references use their `@id`.

In [ ]:
# EDA using a numeric field (by @id) from the first record set

if record_sets_ids:
    df = dataframes.get(example_rs_id)
    if df is not None and not df.empty:
        # Try to pick a likely numeric field. We'll scan for one.
        numeric_candidates = df.select_dtypes(include=['number']).columns
        if len(numeric_candidates) == 0:
            # Try to infer numeric columns from string columns with numeric values
            for col in df.columns:
                try:
                    if pd.api.types.is_numeric_dtype(pd.to_numeric(df[col], errors='coerce')):
                        numeric_candidates = pd.Index([col])
                        break
                except:
                    continue
        if len(numeric_candidates):
            numeric_field = numeric_candidates[0]  # Use the first available
            print(f"Numeric field selected (@id): {numeric_field}")
            # Basic filtering
            threshold = df[numeric_field].quantile(0.50)
            filtered_df = df[df[numeric_field] > threshold].copy()
            print(f"Filtered records ({numeric_field} > {threshold:.2f}) count: {len(filtered_df)}")

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by another field if present (e.g., 'Description' or first string field)
            str_cols = df.select_dtypes(include=['object']).columns
            group_field = None
            for c in str_cols:
                if c != numeric_field and df[c].nunique() < len(df) // 2:
                    group_field = c
                    break

            if group_field:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
                print(grouped_df.head())
            else:
                print("No suitable group field found for grouping.")
        else:
            print("No numeric fields found for EDA in record set.")
    else:
        print("No data found for EDA in the record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot of selected numeric field
if record_sets_ids and df is not None and not df.empty and len(numeric_candidates):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field])
    plt.title(f"Boxplot of '{numeric_field}'")
    plt.tight_layout()
    plt.show()

# If a group field was found, plot group means as barplot
if 'group_field' in locals() and group_field is not None:
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"Mean of '{numeric_field}' by '{group_field}'")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded and explored the FAIR^2 mass spectrometry extracellular vesicle dataset via the Croissant schema using `mlcroissant`.
- All references to dataset elements were made via their stable `@id` URIs.
- After extracting the record sets and fields, we performed basic exploratory operations, such as numeric filtering, normalization, and grouping.
- Standard visualizations were applied, offering insight into distribution and group-level patterns.

This workflow can be extended to downstream analysis, machine learning, or integration with other FAIR datasets using the Croissant format.